# B1 on Colab's free T4 — Qwen2.5-7B-Instruct, 4-bit

**Explainable Financial RAG.** Runs the heavy generation passes on a free GPU so your laptop does none of it.

**This costs ₹0.** Free-tier Colab, open weights from HuggingFace, no API key anywhere. If any cell ever asks for a paid key, stop — something is wrong.

### Before you start
`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

### Storage
About **150 MB in Google Drive** — code, datasets, indices and results. That is 1% of a free 15 GB Drive, so no account juggling is needed.

Model weights are a separate matter: ~15 GB for 7B, downloaded to **Colab's own disk**, never to Drive, and wiped when the session ends. Colab gives ~78 GB there, so it fits — but it re-downloads after a disconnect. If that becomes painful, switch `MODEL` to `Qwen/Qwen2.5-3B-Instruct` in the config cell; the download drops to ~6 GB.

### It is safe to be disconnected
Free Colab drops sessions, usually after a few hours. Everything except the model cache lives in Drive, and every run checkpoints after each batch of 10 questions. If you get disconnected, reconnect and **run all cells again** — completed questions are skipped and the run resumes where it stopped. Nothing is lost.

### Roughly how long

| Step | Time on a T4 |
|---|---|
| Setup + 7B model download | ~15–25 min (repeats after a disconnect) |
| Prompt ablation, 4 arms × 50 questions | ~35 min |
| FinQA, 250 questions | ~45 min |
| TAT-QA, 250 questions | ~45 min |

You do not have to do it in one sitting. Each of the last three cells is independently resumable.

## 1. Confirm we actually have a GPU

4-bit loading is CUDA-only. Failing here with a clear message beats failing 20 minutes into a run.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun this cell."
)
name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name}  ({total_gb:.1f} GB)")
print(f"torch {torch.__version__}")
if total_gb < 14:
    print("\nWARNING: under 14GB. 7B in 4-bit may OOM -- fall back to Qwen/Qwen2.5-3B-Instruct"
          " in the config cell below.")

## 2. Mount Drive and get the repo

The project lives in **Google Drive**, not in Colab's local disk. Colab wipes local disk on disconnect; Drive does not. That single choice is what makes checkpoints, the FAISS index and results survive the disconnects free Colab is prone to.

The next cell clones from GitHub on the first run and pulls on every run after, so the Colab copy always matches your laptop. Nothing is uploaded by hand.

If the repo is private, the clone will ask for credentials — make it public, or paste a GitHub personal access token when prompted.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

GITHUB_URL = 'https://github.com/vedh-vishnu-pogakula/finrag-explain.git'

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
REPO = DRIVE / 'finrag-explain'


def run(*cmd, cwd=None, check=True):
    """Run a git command and surface its output -- a silent failure here wastes an hour."""
    done = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (done.stdout + done.stderr).strip()
    if out:
        print(out)
    if check and done.returncode != 0:
        raise SystemExit(f'FAILED: {" ".join(cmd)}')
    return done.returncode


if (REPO / '.git').exists():
    print('Repo already in Drive -- syncing to origin/master.\n')
    run('git', 'fetch', 'origin', cwd=REPO)
    # Hard reset rather than pull. Every run of this notebook writes results into the repo,
    # and those result summaries are version-controlled, so the Drive checkout is reliably
    # dirty and a plain `git pull` refuses. Nothing of value is lost: the laptop is the
    # source of truth for code, and every result here is reproducible from the checkpoints.
    #
    # NOT `git clean`: eval/results/checkpoints/ is gitignored and holds the per-question B1
    # records. Deleting those means re-running two hours of generation, so untracked files
    # are left exactly where they are.
    run('git', 'reset', '--hard', 'origin/master', cwd=REPO)
else:
    if REPO.exists():
        raise SystemExit(
            f'{REPO} exists but is not a git checkout. Rename or delete it in Drive, '
            'then re-run this cell.'
        )
    print(f'Cloning {GITHUB_URL} -> {REPO} (first run only)\n')
    run('git', 'clone', GITHUB_URL, str(REPO))

os.chdir(REPO)
assert (REPO / 'src' / 'generation' / 'calculator.py').exists(), (
    'Repo looks incomplete -- did the push include the new generation files?'
)
print(f'\nWorking directory: {os.getcwd()}')

# The B1 per-question records are what B2 decomposition and all grounding read. They live
# only in Drive (gitignored, 5MB), so losing them costs a full re-run -- worth confirming.
ckpts = sorted((REPO / 'eval' / 'results' / 'checkpoints').glob('b1_rag_*_dev.jsonl'))
print('B1 checkpoints preserved:', [f'{p.name} ({sum(1 for _ in open(p))} records)'
                                    for p in ckpts] or 'NONE -- B1 must be re-run')

drive_usage = shutil.disk_usage('/content/drive/MyDrive')
local = shutil.disk_usage('/content')
print(f'\nDrive free:       {drive_usage.free / 1024**3:6.1f} GB   '
      f'(this project needs ~0.15 GB)')
print(f'Colab disk free:  {local.free / 1024**3:6.1f} GB   '
      f'(model download lands here, ~15 GB for 7B)')

## 3. Install dependencies

Colab's own CUDA build of `torch` is kept — see the comments in `requirements-colab.txt` for why the pin is not forced here. The exact versions that resolve are recorded in step 4 so any number produced on Colab can be traced back to its environment.

In [ ]:
%%capture install_log
# Everything from the pinned local environment except torch (Colab's CUDA build stays).
!grep -vE '^\s*(#|$)|^torch==' requirements.txt > /tmp/req-colab-core.txt
!pip install -q -r /tmp/req-colab-core.txt
!pip install -q -r requirements-colab.txt
!python -m spacy download en_core_web_sm

In [ ]:
# Surface only the failures -- the full pip log is in `install_log` if you need it.
problems = [ln for ln in install_log.stdout.splitlines()
            if 'ERROR' in ln or 'incompatible' in ln]
print('\n'.join(problems) if problems else 'Install clean.')

import bitsandbytes, spacy, transformers  # noqa: F401
print(f'transformers {transformers.__version__}  bitsandbytes {bitsandbytes.__version__}')

## 4. Record the environment

A result produced in an unrecorded environment is not reproducible. This writes the resolved versions next to the results, so a Colab number and a laptop number can always be told apart later.

In [ ]:
import json
import platform
import sys
from importlib.metadata import PackageNotFoundError, version

packages = ['torch', 'transformers', 'bitsandbytes', 'accelerate',
            'sentence-transformers', 'faiss-cpu', 'spacy', 'numpy']


def _version(pkg):
    """Record a missing package as missing rather than crashing the cell."""
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


env = {
    'platform': platform.platform(),
    'python': sys.version.split()[0],
    'gpu': torch.cuda.get_device_name(0),
    'packages': {p: _version(p) for p in packages},
}
out = Path('eval/results/colab_env.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

## 5. Datasets

Downloaded into Drive, so this is a one-time cost. Both are public and free.

In [ ]:
if Path('data/raw/finqa/dev.json').exists() and Path('data/raw/tatqa/dev.json').exists():
    print('Datasets already in Drive -- skipping download.')
else:
    !bash scripts/download_data.sh

## 6. Point the config at the 7B model

`local_model`, `use_program`, `n_shot` and `load_in_4bit` together are the **frozen generation configuration**. Every baseline — B1 now, B2 and B3 later — must be produced under exactly this setting, or the comparison between them means nothing. Change it here and you have to re-run all three.

In [ ]:
import yaml
from pathlib import Path

# ---------------------------------------------------------------------------------
# This cell CHECKS the frozen generation configuration; it does not write it.
#
# It used to overwrite configs/config.yaml with yaml.safe_dump on every run, which had
# two bad effects: safe_dump strips every comment from the file -- and those comments
# carry the guardrails -- and rewriting a tracked file left the Drive checkout dirty, so
# the next session's git pull refused to run.
#
# configs/config.yaml in the repo is now the single source of truth. To change the frozen
# configuration, change it on the laptop and push. Overrides for a one-off run belong on
# the command line (--model / --no-4bit), never in the file.
# ---------------------------------------------------------------------------------
EXPECTED = {
    'local_model': 'Qwen/Qwen2.5-7B-Instruct',
    'load_in_4bit': True,
    'use_program': True,
    'n_shot': 3,
}

cfg = yaml.safe_load(Path('configs/config.yaml').read_text())['generation']
print(yaml.safe_dump({k: cfg.get(k) for k in EXPECTED}, sort_keys=False))

mismatched = {k: (v, cfg.get(k)) for k, v in EXPECTED.items() if cfg.get(k) != v}
if mismatched:
    for key, (want, got) in mismatched.items():
        print(f'MISMATCH {key}: expected {want!r}, config says {got!r}')
    raise SystemExit(
        'The frozen configuration does not match what this notebook expects. Fix '
        'configs/config.yaml on the laptop and push, rather than editing it here -- '
        'B1, B2 and B3 must all be produced under the same setting.'
    )
print('Frozen configuration confirmed. B1/B2/B3 are comparable.')

# If the T4 cannot fit 7B in 4-bit, do NOT edit the file here -- change it on the laptop,
# push, and re-run. That keeps the config that produced the numbers under version control.

## 7. Smoke test — 5 questions

Downloads the weights (~5GB, once) and proves the whole path works before committing to a two-hour run. Check that `program:` is well above 0 and `unparseable_json` is near 0; those two say the program-of-thought prompt took effect.

In [ ]:
!python eval/baselines/run_b1_rag.py --dataset finqa --split dev --limit 5 --tag _COLAB-SMOKE --fresh 2>&1 | grep -vE 'Batches:|^\s*$' | tail -20

## 8. Prompt ablation at 7B

The same four-way comparison already measured at 1.5B, repeated at 7B. This is the cell that turns "program-of-thought helped our small model" into a claim about the prompt rather than about the model — if the ordering holds at both sizes, the finding replicates.

Resumable: rerun after a disconnect and finished arms are skipped.

In [ ]:
!python eval/baselines/run_prompt_ablation.py --dataset finqa --limit 50 --model Qwen/Qwen2.5-7B-Instruct --load-in-4bit --tag-prefix 7b 2>&1 | grep -vE 'Batches:|^\s*$'

## 9. Full B1 — FinQA, 250 questions

The headline number. ~45 min; checkpoints every 10 questions.

In [ ]:
!python eval/baselines/run_b1_rag.py --dataset finqa --split dev 2>&1 | grep -vE 'Batches:|^\s*$' | tail -25
# Re-score from the saved predictions with the CURRENT metrics. Free, no GPU, no generation.
# Needed because the checkpoints in Drive were scored before FinQA moved to execution accuracy
# against exe_ans; without this the report would be rewritten with the superseded verdicts.
!python eval/baselines/run_b1_rag.py --dataset finqa --split dev --rescore 2>&1 | grep -vE 'Batches:|^\s*$' | tail -14

## 10. Full B1 — TAT-QA, 250 questions

In [ ]:
!python eval/baselines/run_b1_rag.py --dataset tatqa --split dev 2>&1 | grep -vE 'Batches:|^\s*$' | tail -25
!python eval/baselines/run_b1_rag.py --dataset tatqa --split dev --rescore 2>&1 | grep -vE 'Batches:|^\s*$' | tail -14

## 10b. B2 — RAGAS faithfulness, decomposition phase (Month 5)

B2 is B1 plus RAGAS faithfulness. The metric runs in two stages with very different costs, and only the first needs this GPU:

| Stage | Reads | Runs on | Cost |
|---|---|---|---|
| **decompose** | question + answer | the 7B judge | ~500 calls, **do it here** |
| verify | statements + context | a local NLI model | free, run it on your laptop |

Decomposition never reads the retrieved context, so its output cannot change when Month 6 perturbs that context. Caching it here is what makes the Month 6 perturbation sweep cost nothing — otherwise it would be thousands of LLM calls recomputing an identical result.

Resumable like everything else: the statement cache is keyed by question and checkpoints every 10.

**No paid API can be reached.** The judge is constructed explicitly and every call runs inside a guard that strips paid credentials and substitutes an invalid key — see `src/faithfulness/ragas_local.py` for why that matters (RAGAS's own factory defaults to `provider="openai"`).

In [ ]:
!python eval/baselines/run_b2.py --dataset finqa --split dev --phase decompose 2>&1 | grep -vE 'Batches:|^\s*$' | tail -12
!python eval/baselines/run_b2.py --dataset tatqa --split dev --phase decompose 2>&1 | grep -vE 'Batches:|^\s*$' | tail -12

## 10c. Verifier cross-check — is the substitution defensible?

B2 deviates from RAGAS's default and that has to be defended with evidence, not asserted.

RAGAS's `FaithfulnesswithHHEM` replaces the LLM verifier with Vectara's HHEM model. It **cannot run on this stack** — HHEM ships custom remote code predating `transformers` 5.x and dies with `AttributeError: ... 'all_tied_weights_keys'`; downgrading transformers is not an option because 5.15.0 produced every B1 number. So stage 2 runs a standard NLI cross-encoder instead.

The swap is one RAGAS itself sanctions. The open question is whether *this particular* verifier reaches the same verdicts as RAGAS's LLM verifier **on this data**. This cell measures that on 50 questions and reports:

- **score agreement** — Pearson/Spearman and mean |difference| between the two per-question scores
- **verdict agreement** — statement-level accuracy and **Cohen's kappa**, because raw agreement flatters any pair on skewed data
- **the actual disagreements**, with the LLM's stated reason, so the failure mode is inspectable

Costs one LLM call per question — affordable on 50, which is exactly why the NLI verifier exists for the 250-question runs and Month 6's perturbation sweep.

In [ ]:
!python eval/baselines/run_verifier_crosscheck.py --dataset finqa --split dev --limit 50 2>&1 | grep -vE 'Batches:|^\s*$' | tail -22
!python eval/baselines/run_verifier_crosscheck.py --dataset tatqa --split dev --limit 50 2>&1 | grep -vE 'Batches:|^\s*$' | tail -22

## 10d. B2 under RAGAS's *unmodified* LLM verifier — the decisive run

Cell 10c compared the two verifiers on 50 questions and found they agree at **chance** (Cohen's kappa −0.147 FinQA, +0.157 TAT-QA). It left one question underpowered: does RAGAS's own LLM verifier separate *provably grounded* answers from ungrounded ones? On 50 questions only 9 FinQA cases and **1** TAT-QA case were ungrounded — far too few.

This cell runs the LLM verifier over all 250 questions per dataset. It gives two things the paper needs:

1. **B2 under RAGAS's default metric**, unmodified — not our NLI substitute.
2. The full-power test against operand provenance, a deterministic reference that involves no model judgement at all.

~1 call per question, checkpointed and resumable. **This is the last heavy run for Month 5.** Month 6's perturbation sweep needs no GPU at all, because it re-uses the cached statements and the local NLI verifier.

In [ ]:
!python eval/baselines/run_b2.py --dataset finqa --split dev --phase verify --verifier llm --out-tag _LLMVER 2>&1 | grep -vE 'Batches:|^\s*$' | tail -16
!python eval/baselines/run_b2.py --dataset tatqa --split dev --phase verify --verifier llm --out-tag _LLMVER 2>&1 | grep -vE 'Batches:|^\s*$' | tail -16

## 10e. Is the LLM verifier's advantage specific to Qwen?

The Month 5 result is that RAGAS's LLM verifier separates correct from incorrect answers (+0.25 FinQA, +0.24 TAT-QA, both p<0.001) while NLI verifiers do not. But **the same model family — Qwen2.5-7B — does the decomposition *and* the LLM verification**, so a reviewer can reasonably ask whether the advantage is a property of LLM verification or of that one model.

This cell re-runs stage-2 verification with a judge from a different family, reusing the identical cached statements. If the advantage holds, the finding is about verifier *architecture*; if it vanishes, it is about Qwen, and the paper must say so.

**On model choice:** many strong 7–8B instruct models are gated on HuggingFace and will fail to download without an accepted licence and a token. `microsoft/Phi-3.5-mini-instruct` is ungated and capable. If you have a HF token and have accepted the licence, `meta-llama/Llama-3.1-8B-Instruct` is the closer size match — set `JUDGE` below.

~500 calls, checkpointed and resumable.

In [ ]:
import sys
JUDGE = 'microsoft/Phi-3.5-mini-instruct'   # ungated; swap for Llama-3.1-8B if you have a token
TAG = '_LLMVER-ALT'

import subprocess
for ds in ['finqa', 'tatqa']:
    print(f'\n===== {ds} =====')
    subprocess.run([sys.executable, 'eval/baselines/run_b2.py', '--dataset', ds, '--split', 'dev',
                    '--phase', 'verify', '--verifier', 'llm', '--model', JUDGE,
                    '--out-tag', TAG])

## 10f. Perturbation audit under RAGAS's LLM verifier

Cell 10 (local) ran the audit with the NLI verifier and found it **passes**: removing the operand-supplying sentence hurts significantly more than removing an equal number of unrelated sentences (+0.087 FinQA, +0.228 TAT-QA).

That completes only half the 2×2. This cell runs the same three conditions under the LLM verifier, which answers the question the paper needs:

> Is the better verifier *also* the more evidence-specific one, or do both respond to perturbation equally while only one tracks correctness?

If both are evidence-specific but only the LLM verifier separates correct from incorrect, that is the cleanest possible statement of the paper's methodological point: **perturbation sensitivity is necessary but not sufficient for metric validity.**

Three judge calls per question, so this runs on a subset. Checkpointed by (question, condition) — a disconnect costs at most one call.

In [ ]:
import sys
import subprocess
for ds in ['finqa', 'tatqa']:
    print(f'\n===== {ds} =====')
    subprocess.run([sys.executable, 'eval/baselines/run_perturbation.py', '--dataset', ds,
                    '--split', 'dev', '--verifier', 'llm', '--limit', '80',
                    '--out-tag', '_LLMVER'])

## 11. Summary

Everything below is already saved in Drive under `eval/results/`. Nothing needs downloading.

In [ ]:
import json
from pathlib import Path

print('=== B1 end-to-end (250 questions each) ===')
for ds in ['finqa', 'tatqa']:
    p = Path(f'eval/results/b1_rag_{ds}_dev.json')
    if not p.exists():
        print(f'{ds}: not run yet')
        continue
    r = json.loads(p.read_text())
    a, c = r['answers'], r['config']
    print(f"\n{ds}  [{c['generation_model']}, {c.get('quantization')}, {c['prompt_version']}]")
    print(f"  numeric_accuracy = {a['numeric_accuracy']}   span_f1 = {a['span_token_f1']}")
    print(f"  recall@5 = {r['overall'].get('recall@5')}   citation_precision = "
          f"{r['citation_precision']}")
    print(f"  program_rate = {r['program_rate']}   unparseable = {r['unparseable_json_rate']}")
    print(f"  failures = {r['failure_modes']}")

abl = Path('eval/results/ablation_finqa_dev_7b.md')
if abl.exists():
    print('\n' + abl.read_text())

## Done

Results are in Drive at `finrag-explain/eval/results/`:

| File | What it is |
|---|---|
| `b1_rag_{finqa,tatqa}_dev.json` | headline B1 numbers |
| `checkpoints/b1_rag_*.jsonl` | every question, its evidence, expression and score — this is what the failure taxonomy is built from |
| `ablation_finqa_dev_7b.{json,md}` | the 7B prompt ablation table |
| `colab_env.json` | the environment these numbers came from |

To bring them back to the laptop, copy the `eval/results/` folder out of Drive — the per-question `.jsonl` files matter as much as the summaries, since the failure analysis is done on them.

**Next:** Month 5 — the evidence-grounding layer and RAGAS (B2). Before writing any B2 code, read the RAGAS note in `CLAUDE.md`: it defaults to an OpenAI judge and bills silently on the first call, so the judge has to be set explicitly to a local or free model *before* that call, not after.

## 12. Artifact manifest — did this session actually produce what it should?

A stale browser tab runs whatever cell list it loaded, reports no errors, and writes nothing new. That is indistinguishable from success unless something checks. This cell lists every artifact the notebook is supposed to produce and flags what is missing, so the run either proves itself or tells you which cell to re-run.

In [ ]:
from pathlib import Path

BASE = Path('/content/drive/MyDrive/finrag-explain/eval/results')
EXPECTED = {
    'cell 9/10  B1':        [f'b1_rag_{d}_dev.json' for d in ('finqa', 'tatqa')],
    'cell 8     ablation':  ['ablation_finqa_dev_7b.json'],
    'cell 10b   statements':[f'checkpoints/statements_{d}_dev.jsonl' for d in ('finqa','tatqa')],
    'cell 10c   crosscheck':[f'verifier_crosscheck_{d}_dev.json' for d in ('finqa','tatqa')],
    'cell 10d   LLM verify':[f'b2_{d}_dev_LLMVER.json' for d in ('finqa','tatqa')],
    'cell 10e   alt judge': [f'b2_{d}_dev_LLMVER-ALT.json' for d in ('finqa','tatqa')],
    'cell 10f   LLM perturb':[f'perturbation_{d}_dev_LLMVER.json' for d in ('finqa','tatqa')],
}

missing = []
for label, files in EXPECTED.items():
    for f in files:
        path = BASE / f
        ok = path.exists()
        size = f'{path.stat().st_size/1024:7.1f} KB' if ok else '   ABSENT'
        print(f"{'OK ' if ok else '** '} {label:24s} {size}  {f}")
        if not ok:
            missing.append((label, f))

if missing:
    cells = sorted({m[0].split()[1] for m in missing})
    print(f'\n{len(missing)} artifact(s) missing -- re-run cell(s): {", ".join(cells)}')
    print('If a whole cell is missing from this notebook, your browser copy is stale:')
    print('  File -> Revert to saved version, then Runtime -> Run all.')
else:
    print('\nAll expected artifacts present. Download them and continue on the laptop.')